# View the downloaded DistilBERT results

Put `wine_distilbert_results.zip`, `extract_distilbert_results.py`, and this notebook
in the project folder. Open the notebook from that folder and run the cells in order.
The first cell extracts the ZIP into `wine_distilbert_results/` and lists every file.
An existing extraction can be reused when the files match; differing files are not overwritten.

The remaining cells show the saved reports, CSV tables, and a classification report
recomputed from the validation predictions. Model files are listed with their sizes.
This notebook does not load the model or train it. It requires pandas and scikit-learn,
which are also used by the project's baseline notebooks.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report, f1_score,
)

PROJECT_DIR = Path.cwd().resolve()
ARCHIVE_PATH = PROJECT_DIR / "wine_distilbert_results.zip"
OUTPUT_DIR = PROJECT_DIR / "wine_distilbert_results"

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(
        f"Place wine_distilbert_results.zip in {PROJECT_DIR}, then run this cell again."
    )

from extract_distilbert_results import extract_results

extracted_files = sorted(extract_results(ARCHIVE_PATH, OUTPUT_DIR))
print(f"Results folder: {OUTPUT_DIR}")
print(f"Exported files: {len(extracted_files)}")
file_inventory = pd.DataFrame([
    {
        "file": path.relative_to(OUTPUT_DIR).as_posix(),
        "bytes": path.stat().st_size,
        "MiB": round(path.stat().st_size / 2**20, 3),
    }
    for path in extracted_files
])
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(file_inventory)

Results folder: C:\Users\dilra\dataprocess\4401project3\wine_distilbert_results
Exported files: 12


,file,bytes,MiB
0,confusion_matrix.csv,237,0.000
1,metrics.json,304,0.000
2,model/config.json,730,0.001
3,model/model.safetensors,267841796,255.434
4,model/special_tokens_map.json,125,0.000
5,model/tokenizer.json,711494,0.679
6,model/tokenizer_config.json,1227,0.001
7,model/training_args.bin,5841,0.006
8,model/vocab.txt,231508,0.221
9,training_history.json,31643,0.030


In [2]:
# Show every report JSON in full, including metrics, settings, and history if present.
# Tokenizer/model JSON files are included in the inventory instead of being printed.
report_json_files = [
    path for path in extracted_files
    if path.suffix.lower() == ".json"
    and "model" not in path.relative_to(OUTPUT_DIR).parts
]
if not report_json_files:
    print("No report JSON files were found in the archive.")

json_reports = {}
for path in report_json_files:
    relative_name = path.relative_to(OUTPUT_DIR).as_posix()
    data = json.loads(path.read_text(encoding="utf-8"))
    json_reports[relative_name] = data
    display(Markdown(f"### {relative_name}"))
    print(json.dumps(data, indent=2, ensure_ascii=False))

### metrics.json

{
  "validation_loss": 0.6298307776451111,
  "validation_accuracy": 0.9582976208068568,
  "validation_balanced_accuracy": 0.8289458930786726,
  "validation_macro_f1": 0.81427302134862,
  "validation_runtime": 25.8147,
  "validation_samples_per_second": 1310.688,
  "validation_steps_per_second": 40.984
}


### training_history.json

[
  {
    "loss": 1.6016,
    "grad_norm": 3.84072208404541,
    "learning_rate": 1.0399159663865548e-06,
    "epoch": 0.015760441292356184,
    "step": 100
  },
  {
    "loss": 1.5661,
    "grad_norm": 4.871952533721924,
    "learning_rate": 2.090336134453782e-06,
    "epoch": 0.03152088258471237,
    "step": 200
  },
  {
    "loss": 1.5158,
    "grad_norm": 4.278702259063721,
    "learning_rate": 3.1407563025210084e-06,
    "epoch": 0.04728132387706856,
    "step": 300
  },
  {
    "loss": 1.4956,
    "grad_norm": 5.944331169128418,
    "learning_rate": 4.191176470588236e-06,
    "epoch": 0.06304176516942474,
    "step": 400
  },
  {
    "loss": 1.5278,
    "grad_norm": 5.376126766204834,
    "learning_rate": 5.241596638655463e-06,
    "epoch": 0.07880220646178093,
    "step": 500
  },
  {
    "loss": 1.3698,
    "grad_norm": 15.024679183959961,
    "learning_rate": 6.29201680672269e-06,
    "epoch": 0.09456264775413711,
    "step": 600
  },
  {
    "loss": 1.1207,
    "grad_norm": 4

### training_settings.json

{
  "model": "distilbert/distilbert-base-uncased",
  "max_length": 128,
  "label2id": {
    "red": 0,
    "white": 1,
    "sparkling": 2,
    "rose": 3,
    "dessert": 4
  },
  "class_weights": [
    0.3259090483188629,
    0.6236927509307861,
    6.111137866973877,
    8.21578311920166,
    23.281192779541016
  ],
  "training_arguments": {
    "output_dir": "wine_distilbert_checkpoints",
    "overwrite_output_dir": false,
    "do_train": false,
    "do_eval": true,
    "do_predict": false,
    "eval_strategy": "epoch",
    "prediction_loss_only": false,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "per_gpu_train_batch_size": null,
    "per_gpu_eval_batch_size": null,
    "gradient_accumulation_steps": 1,
    "eval_accumulation_steps": null,
    "eval_delay": 0,
    "torch_empty_cache_steps": null,
    "learning_rate": 2e-05,
    "weight_decay": 0.01,
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_epsilon": 1e-08,
    "max_grad_norm": 1.0,

In [3]:
# Show complete CSV reports and a manageable preview of the validation predictions.
csv_tables = {}
for path in extracted_files:
    if path.suffix.lower() != ".csv":
        continue
    relative_name = path.relative_to(OUTPUT_DIR).as_posix()
    table = pd.read_csv(path, index_col=0 if path.name == "confusion_matrix.csv" else None)
    csv_tables[relative_name] = table
    display(Markdown(f"### {relative_name}"))
    print(f"{len(table):,} rows; {len(table.columns)} columns")
    if path.name == "validation_predictions.csv":
        print("First 10 predictions; the full table is available in csv_tables above.")
        with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
            display(table.head(10))
    else:
        with pd.option_context(
            "display.max_rows", None, "display.max_columns", None,
            "display.max_colwidth", None,
        ):
            display(table)

if not csv_tables:
    print("No CSV files were found in the archive.")

### confusion_matrix.csv

5 rows; 5 columns


,predicted red,predicted white,predicted sparkling,predicted rose,predicted dessert
actual red,20267,140,88,218,51
actual white,102,10461,214,36,37
actual sparkling,28,185,850,38,6
actual rose,72,57,28,662,4
actual dessert,60,37,5,5,184


### validation_predictions.csv

33,835 rows; 4 columns
First 10 predictions; the full table is available in csv_tables above.


,row_id,actual_color,predicted_color,description
0,154558,red,red,"The Big Pig Vineyard, for which this wine is named, is the producer's estate vineyard, a site planted to four Pinot clones. Altogether it's stunning, a vibrant mix of funky lavender and licorice that's big in both flavor and presence on the palate, with a grip of soft tannins. The finish is concentrated in black cherry, with a seasoning of Asian spice."
1,158490,red,red,"Raisin-baked plum defines this hearty, full-bodied and ultraripe wine, rich in red berry and robust on the finish."
2,150281,white,white,"There's a touch of effervescence upon opening this twist-top bottling, which makes the aromas of fresh tangerine peel, pear cider, white rocks and Pink Lady apple skins very accessible. It's extremely racy and zippy on the sip, with tons of citrus blossom and ripe stone-fruit flavors, all cut by a slightly sour mineral-laced acidity."
3,16898,red,red,"Dark and forceful, with an earthy nose of tar, leather and baked fruit. The palate has life and size, but also quasi-sharp acidity, while flavors of blackberry carry a lactic, vanilla undertone. Finishes sweet but a little burnt."
4,143787,white,white,"This is a young wood-aged wine with rich toast. While the toast does smother the fruit at the moment, there are enough crisp acidity and citrus flavors in the background to come through as the wine matures for another year or two. Drink from 2018."
5,122202,rose,rose,"This is an extremely textural pink wine, with delicate touches of raspberry, strawberry, watermelon and the slightest twist of lime on the nose. The blend of 68% Grenache, 13% Counoise, 11% Mourvèdre and 8% Syrah shows grip and acidity on the sip, where lime-soaked nectarine and red apple flavors reign."
6,115756,red,red,"This relative newcomer from near Pisa is three-fourths Sangiovese and the remainder Merlot, and the mix is delightful. Black-plum aromas are hardened by the background scents of tar and leather, while soft, smooth tannins allow for a chocolaty, soft mouthfeel that steals the show. Rich, sweet and smooth."
7,99086,white,white,"Full-on style of wine, packed with apricot and peach flavors, spiced up with cinnamon, layered with toast and finishing with delicious, open acidity. A wine that shows why St-Aubin is a village that is worth paying attention to."
8,122549,red,red,"The aromas bring notes of herb, sweet tobacco and ash. The plum flavors are tart and full in feel, with the tannins giving a (quite) chalky squeeze."
9,152049,red,red,"A taste of juicy red cherry provides a brightness of acidity to this wine, a 100% varietal aged in entirely French oak. Soft, integrated tannins form a foundation around dried herbal notes and black pepper, working seamlessly together to an elegant end."


In [4]:
# Recompute validation scores directly from the saved predictions.
prediction_matches = [
    (name, table) for name, table in csv_tables.items()
    if Path(name).name == "validation_predictions.csv"
]
if not prediction_matches:
    print("No validation_predictions.csv was found, so no classification report can be calculated.")
elif len(prediction_matches) != 1:
    raise ValueError("More than one validation_predictions.csv was found; choose the intended result.")
else:
    prediction_name, predictions = prediction_matches[0]
    required_columns = {"actual_color", "predicted_color"}
    if not required_columns.issubset(predictions.columns):
        raise ValueError(f"{prediction_name} must contain actual_color and predicted_color.")
    if predictions.empty or predictions[list(required_columns)].isna().any().any():
        raise ValueError("The prediction table is empty or contains missing labels.")

    COLORS = ["red", "white", "sparkling", "rose", "dessert"]
    actual = predictions["actual_color"]
    predicted = predictions["predicted_color"]
    unknown_labels = (set(actual) | set(predicted)) - set(COLORS)
    if unknown_labels:
        raise ValueError(f"Unexpected wine categories: {sorted(unknown_labels)}")

    display(Markdown("### Validation classification report"))
    print(classification_report(
        actual, predicted, labels=COLORS, target_names=COLORS,
        digits=4, zero_division=0,
    ))
    validation_scores = pd.Series({
        "accuracy": accuracy_score(actual, predicted),
        "balanced_accuracy": balanced_accuracy_score(actual, predicted),
        "macro_f1": f1_score(
            actual, predicted, labels=COLORS, average="macro", zero_division=0,
        ),
    }, name="recomputed_validation_score")
    display(validation_scores.to_frame())

### Validation classification report

              precision    recall  f1-score   support

         red     0.9872    0.9761    0.9816     20764
       white     0.9615    0.9641    0.9628     10850
   sparkling     0.7173    0.7678    0.7417      1107
        rose     0.6903    0.8044    0.7430       823
     dessert     0.6525    0.6323    0.6422       291

    accuracy                         0.9583     33835
   macro avg     0.8018    0.8289    0.8143     33835
weighted avg     0.9600    0.9583    0.9590     33835



,recomputed_validation_score
accuracy,0.958298
balanced_accuracy,0.828946
macro_f1,0.814273
